# 00 — Kurulum kontrolü (smoke test)

Her ekip üyesi kurulumdan sonra bu notebook'u çalıştırır. **Üç makinede de**
split boyutları ve `splits` özeti aynı çıkmalıdır; aynı değilse kimse model
eğitmeye başlamasın.

In [1]:
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [2]:
import importlib
import platform

print("python         ", platform.python_version())
for name in ["numpy", "pandas", "sklearn", "imblearn", "sdv", "lightgbm", "optuna",
             "torch", "pytorch_tabnet", "missingno", "matplotlib", "seaborn"]:
    module = importlib.import_module(name)
    print(f"{name:15s} {getattr(module, '__version__', 'ok')}")

python          3.10.11
numpy           2.2.6


pandas          2.3.3


sklearn         1.7.2


imblearn        0.14.2


sdv             2.0.0
lightgbm        4.7.0
optuna          5.0.0
torch           2.14.1+cpu
pytorch_tabnet  ok


missingno       0.5.2
matplotlib      3.10.9
seaborn         0.13.2


In [3]:
import pandas as pd

from src.config import TARGET
from src.data import load_split, load_splits

splits = load_splits()
# Content hash (not file bytes), so line endings cannot change it.
print("splits özeti:", pd.util.hash_pandas_object(splits, index=False).sum())
pd.DataFrame(
    {s: {"rows": len(d), "default_rate": round(d[TARGET].mean(), 4)}
     for s in ("train", "val", "test") for d in [load_split(s, "v0")]}
).T

splits özeti: 6552357087371317484


,rows,default_rate
train,20975.0,0.2213
val,4495.0,0.2211
test,4495.0,0.2214


In [4]:
from src.data import to_xy
from src.features import make_preprocessor

X, y = to_xy(load_split("train", "v0"))
{kind: make_preprocessor(kind).fit_transform(X).shape for kind in ("linear", "tree", "tabnet")}

{'linear': (20975, 33), 'tree': (20975, 23), 'tabnet': (20975, 23)}